In [1]:
import re
from urllib.parse import urlparse
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# 1. Feature Extraction Function
def extract_features(url):
    """Extracts numerical features from a given URL string."""
    features = []

    # Feature 1: Length of URL
    features.append(len(url))

    # Feature 2: Presence of IP address (1 if IP present, else 0)
    ip_pattern = re.compile(
        r'(([0-9]|[1-9][0-9]|1[0-9]{2}|2[0-4][0-9]|25[0-5])\.){3}'
        r'([0-9]|[1-9][0-9]|1[0-9]{2}|2[0-4][0-9]|25[0-5])'
    )
    features.append(1 if ip_pattern.search(url) else 0)

    # Feature 3: Presence of '@' symbol (often used to obscure real domain)
    features.append(1 if '@' in url else 0)

    # Feature 4: Number of dots '.'
    features.append(url.count('.'))

    # Feature 5: Number of hyphens '-' (common in deceptive phishing domains)
    features.append(url.count('-'))

    # Feature 6: Number of slashes '/'
    features.append(url.count('/'))

    # Feature 7: Use of HTTPS protocol (1 if https, 0 otherwise)
    parsed_url = urlparse(url)
    features.append(1 if parsed_url.scheme == 'https' else 0)

    return features


# 2. In-Memory Training Dataset (No Database Required)
# Labels: 0 -> Legitimate, 1 -> Phishing
training_data = [
    ("https://www.google.com", 0),
    ("https://github.com", 0),
    ("https://www.wikipedia.org", 0),
    ("https://www.python.org", 0),
    ("http://secure-login-update-apple.com/signin", 1),
    ("http://192.168.1.1/update/verify", 1),
    ("http://paypal-account-security-check.com", 1),
    ("http://verify-bank-login-info.com/signin", 1)
]

# Separate URLs and labels
X_raw = [extract_features(url) for url, label in training_data]
y_train = [label for url, label in training_data]


# 3. Feature Scaling & SVM Model Training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_raw)

# Initialize and train the Support Vector Machine classifier
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_scaled, y_train)


# 4. Prediction Function for Testing New URLs
def predict_phishing_url(url):
    features = extract_features(url)
    # Scale features using the same scaler fitted on training data
    features_scaled = scaler.transform([features])

    prediction = svm_model.predict(features_scaled)[0]
    probability = svm_model.predict_proba(features_scaled)[0][prediction]

    label_text = "Phishing 🚨" if prediction == 1 else "Legitimate ✅"
    print(f"URL: {url}")
    print(f"Result: {label_text} (Confidence: {probability * 100:.2f}%)\n")


# --- Execution Example ---
if __name__ == "__main__":
    test_urls = [
        "https://github.com/settings/profile",
        "http://update-paypal-security-alert.com/login",
        "https://www.python.org/downloads/"
    ]

    print("--- Phishing Detection Using SVM (In-Memory) ---\n")
    for test_url in test_urls:
        predict_phishing_url(test_url)

--- Phishing Detection Using SVM (In-Memory) ---

URL: https://github.com/settings/profile
Result: Legitimate ✅ (Confidence: 41.16%)

URL: http://update-paypal-security-alert.com/login
Result: Phishing 🚨 (Confidence: 93.58%)

URL: https://www.python.org/downloads/
Result: Legitimate ✅ (Confidence: 50.00%)

